In [1]:
import sys
import mne
sys.path.append('../')

In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

"""
datasets=[
    BNCI2014_008(),
    BNCI2014_009(),
    BNCI2015_003(),
    BI2012(),
    BI2013a(),
    BI2014a(),
    BI2014b(),
    BI2015a(),
    BI2015b(),
    Cattan2019_VR(),
    EPFLP300(),
    Huebner2017(),
    Huebner2018(),
    Lee2019_ERP()
]
"""

datasets = [BNCI2014_008()]


paradigm = P300(
    resample=48,

)
cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

In [3]:
import copy
from sklearn.base import clone
import dask
import os
from sklearn.preprocessing import FunctionTransformer
import tensorly as tl

def eval_moabb_within_session(dataset, subject, pipe):
    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=False,
        random_state=42,
        n_jobs=5,
        suffix=f'bttda_dask_dataset-{dataset.code}_subject-{subject}_pipe-{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process({pipe:clone(pipelines[pipe])})


In [ ]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pdF
from classification_erp import get_pipelines
from hpc import create_cluster, create_client, TIMEOUT

pipelines = get_pipelines()

with create_cluster(cluster='local') as cluster, create_client(cluster) as client:
    results = []
    for dataset in datasets:
        print(f'Benchmarking on dataset {dataset.code}...')
        job_args = []
        for subject in dataset.subject_list:
            for pipe in pipelines.keys():
                job_args.append((dataset, subject,pipe))    
        with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
            results += Parallel(n_jobs=len(job_args), verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

Benchmarking on dataset BNCI2014-008...


[Parallel(n_jobs=24)]: Using backend DaskDistributedBackend with 12 concurrent workers.
BNCI2014-008-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]

dataset=BNCI2014-008, subject=2/8, pipe=HODA
dataset=BNCI2014-008, subject=3/8, pipe=BTTDA
dataset=BNCI2014-008, subject=8/8, pipe=PARAFACDA
dataset=BNCI2014-008, subject=8/8, pipe=HODA
dataset=BNCI2014-008, subject=7/8, pipe=BTTDA
dataset=BNCI2014-008, subject=7/8, pipe=PARAFACDA
dataset=BNCI2014-008, subject=7/8, pipe=HODA
dataset=BNCI2014-008, subject=6/8, pipe=BTTDA
dataset=BNCI2014-008, subject=6/8, pipe=PARAFACDA
dataset=BNCI2014-008, subject=1/8, pipe=HODA
dataset=BNCI2014-008, subject=5/8, pipe=BTTDA
dataset=BNCI2014-008, subject=5/8, pipe=PARAFACDA
dataset=BNCI2014-008, subject=2/8, pipe=PARAFACDA
dataset=BNCI2014-008, subject=3/8, pipe=HODA


/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock 

dataset=BNCI2014-008, subject=4/8, pipe=HODA


...

/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)




BNCI2014-008-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/

dataset=BNCI2014-008, subject=1/8, pipe=PARAFACDA
dataset=BNCI2014-008, subject=1/8, pipe=BTTDA
dataset=BNCI2014-008, subject=5/8, pipe=HODA
dataset=BNCI2014-008, subject=4/8, pipe=BTTDA







BNCI2014-008-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]





BNCI2014-008-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]






BNCI2014-008-WithinSession:   0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 sec

dataset=BNCI2014-008, subject=3/8, pipe=PARAFACDA
dataset=BNCI2014-008, subject=6/8, pipe=HODA
dataset=BNCI2014-008, subject=8/8, pipe=BTTDA
dataset=BNCI2014-008, subject=2/8, pipe=BTTDA


/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock file after 5 seconds, consider deleting it if you know the corresponding file is usable:
/root/.mne/mne-python.json.lock
  return next(self.gen)
/usr/local/lib/python3.11/contextlib.py:137: RuntimeWarning: Could not acquire lock 

In [ ]:
results.to_csv('results/moabb_erp.csv')
results

In [ ]:
results = pd.read_csv('results/moabb_erp.csv')

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean')

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

In [ ]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[.5, 1])
    fig.update_yaxes(range=[.5, 1])
    fig.add_shape(
        type="line",
        x0=0.5, y0=0.5, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig